## Automated Dataset Acquisition

In [5]:
import os
import pandas as pd
from huggingface_hub import hf_hub_download

# Create local data layout directory if it was cleared
os.makedirs("./data", exist_ok=True)
local_csv_target = "./data/huggingface_comments.csv"

print("Contacting Hugging Face Hub to download textual corpus...")
try:
    # Safely stream down the raw csv dataset from an open-weights community repository
    downloaded_file_path = hf_hub_download(
        repo_id="thesofakillers/jigsaw-toxic-comment-classification-challenge",
        filename="train.csv",
        repo_type="dataset"
    )
    
    # Load a segment of the dataset into pandas to isolate the unstructured text column
    df = pd.read_csv(downloaded_file_path)
    
    # Extract 100 complete text blocks to form your 100-document portfolio baseline
    sample_documents = df["comment_text"].head(100).tolist()
    
    # Merge them into a single string for our chunking processors to scan
    extracted_document_text = "\n\n=== NEW_DOCUMENT_MARKER ===\n\n".join(sample_documents)
    
    print(f"Success! Loaded {len(sample_documents)} real-world text documents from Hugging Face.")
    print(f"Total text length ready for processing: {len(extracted_document_text)} characters.")

except Exception as e:
    print(f"Hugging Face Hub access issue: {e}")


d:\hybrid-retrieval-hnsw-semantic-chunking\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Contacting Hugging Face Hub to download textual corpus...


d:\hybrid-retrieval-hnsw-semantic-chunking\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\windows 10\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\hybrid-retrieval-hnsw-semantic-chunking\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggi

Success! Loaded 100 real-world text documents from Hugging Face.
Total text length ready for processing: 44733 characters.


## Local BGE Model & Algorithmic Semantic Chunking

In [8]:
# Install the foundational packages into your active kernel if missing
!python -m pip install -q sentence-transformers langchain-huggingface langchain-experimental

d:\hybrid-retrieval-hnsw-semantic-chunking\.venv\Scripts\python.exe: No module named pip


In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

print("Initializing BAAI/bge-large-en-v1.5 locally...")

# 1. Spin up the open-source embedding architecture locally
bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cpu'},  # Toggle to 'cuda' if running on a GPU-enabled workspace
    encode_kwargs={'normalize_embeddings': True}
)

print(" Computing moving cosine variance boundaries across text...")

# 2. Inject BGE parameters into the algorithmic text split engine
semantic_processor = SemanticChunker(
    bge_embeddings,
    breakpoint_threshold_type="percentile" # Splits where topic drift triggers a variance drop
)


# Convert the continuous text variable into chunk records
processed_semantic_blocks = semantic_processor.create_documents([extracted_document_text])
semantic_chunk_strings = [chunk.page_content for chunk in processed_semantic_blocks]

print(f" Success! Generated: {len(semantic_chunk_strings)} semantic chunks using BGE.")


C:\Users\windows 10\AppData\Local\Temp\ipykernel_4840\3491145712.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Initializing BAAI/bge-large-en-v1.5 locally...


d:\hybrid-retrieval-hnsw-semantic-chunking\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\windows 10\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:05<00:00, 68.28it/s]


 Computing moving cosine variance boundaries across text...
 Success! Generated: 23 semantic chunks using BGE.


In [12]:
for chunk in semantic_chunk_strings:
    print(chunk)
    print("*"*64)

Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27

=== NEW_DOCUMENT_MARKER ===

D'aww! He matches this background colour I'm seemingly stuck with. Thanks. (talk) 21:51, January 11, 2016 (UTC)

=== NEW_DOCUMENT_MARKER ===

Hey man, I'm really not trying to edit war. It's just that this guy is constantly removing relevant information and talking to me through edits instead of my talk page. He seems to care more about the formatting than the actual info. === NEW_DOCUMENT_MARKER ===

"
More
I can't make any real suggestions on improvement - I wondered if the section statistics should be later on, or a subsection of ""types of accidents""  -I think the references may need tidying so that they are all in the exact same format ie date format etc. I can do that later on, if no-one e

## Offline Local Semantic Chunking set HF token to 1 - cell not executed.

In [ ]:
import os

#  Enforce absolute local offline execution protocol
# Setting HF_HUB_DISABLE_TELEMETRY prevents analytics pings
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
# Setting HF_HUB_OFFLINE forces the hub client to look only at local disk caches
os.environ["HF_HUB_OFFLINE"] = "1"
# Setting TRANSFORMERS_OFFLINE disables external network lookups for weights
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

print(" Initializing BAAI/bge-large-en-v1.5 locally in strict offline mode...")

# Initialize the embedding layer using the pre-downloaded local weights cache
bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cpu'},  # Toggle to 'cuda' if running on a GPU notebook
    encode_kwargs={'normalize_embeddings': True}
)

print(" Computing moving cosine variance boundaries locally...")

# Inject local BGE parameters into the text split engine
semantic_processor = SemanticChunker(
    bge_embeddings,
    breakpoint_threshold_type="percentile"
)

# Segment the memory text variable array
processed_semantic_blocks = semantic_processor.create_documents([extracted_document_text])
semantic_chunk_strings = [chunk.page_content for chunk in processed_semantic_blocks]

print(f" Success! Generated: {len(semantic_chunk_strings)} semantic chunks completely offline.")


## ChromaDB HNSW Skip-Graph Indexing & Optimized Retrieval

In [13]:
import os
import chromadb
from chromadb.utils.embedding_functions import EmbeddingFunction

# 1. Define a clean custom embedding class so Chroma calls our local offline BGE model for query lookups
class BGELocalEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: list[str]) -> list[list[float]]:
        return bge_embeddings.embed_documents(input)

bge_chroma_ef = BGELocalEmbeddingFunction()

# 2. Map the local persistent storage directory for Chroma
chroma_client = chromadb.PersistentClient(path="./local_bge_hnsw_storage")

# 3. Configure the HNSW skip-graph metadata parameters
# This sets up the structural tuning of our multi-layered navigation index layer
hnsw_collection = chroma_client.get_or_create_collection(
    name="huggingface_bge_hnsw_index",
    embedding_function=bge_chroma_ef,
    metadata={
        "hnsw:space": "cosine",       # Cosine similarity metric calculation tracking
        "hnsw:construction_ef": 100,  # High precision index graph build setting
        "hnsw:M": 16                   # Max bi-directional structural links per node
    }
)

# 4. Format and slice chunks for initial population loading
target_chunks = semantic_chunk_strings[:100]  # Limit to 100 entries for portfolio control
document_ids = [f"bge_chunk_{i}" for i in range(len(target_chunks))]

# Assign rich structural tag metadata for our optimized pre-filtering lookup step
metadata_mappings = [
    {"source": "huggingface_toxic_corpus", "doc_type": "text_chunk"} 
    for _ in range(len(target_chunks))
]

print(" Constructing HNSW Graph multi-layer nodes using 1,024-dimensional local BGE vectors...")
hnsw_collection.add(
    documents=target_chunks,
    metadatas=metadata_mappings,
    ids=document_ids
)
print("HNSW Local Vector Skip-Graph Construction Complete!")

# =====================================================================
#  Execute Optimized Query Lookup (Combining Graph Traversal + Metadata Filter)
# =====================================================================
target_user_query = "What are the common patterns or text behaviors extracted from the corpus?"

print(f"\n Executing fast O(log N) HNSW lookup for: '{target_user_query}'...")
query_execution_output = hnsw_collection.query(
    query_embeddings=None,  # Handled natively via the embedding function string wrapper
    query_texts=[target_user_query],
    n_results=2,
    where={"source": "huggingface_toxic_corpus"}  # Hard metadata pre-filtering execution
)

# Render results
print("\n --- Top Retained Search Result ---")
print(query_execution_output['documents'][0][0])


C:\Users\windows 10\AppData\Local\Temp\ipykernel_4840\1337058341.py:10: DeprecationWarning: The class BGELocalEmbeddingFunction does not implement __init__. This will be required in a future version.
  bge_chroma_ef = BGELocalEmbeddingFunction()


 Constructing HNSW Graph multi-layer nodes using 1,024-dimensional local BGE vectors...
HNSW Local Vector Skip-Graph Construction Complete!

 Executing fast O(log N) HNSW lookup for: 'What are the common patterns or text behaviors extracted from the corpus?'...

 --- Top Retained Search Result ---
If you had a better knowledge of the English language it would have been obvious to you. === NEW_DOCUMENT_MARKER ===

Would you both shut up, you don't run wikipedia, especially a stupid kid. === NEW_DOCUMENT_MARKER ===

Oh, it's me vandalising?xD See here. Greetings,

=== NEW_DOCUMENT_MARKER ===

Website 

Hey all,
I was thinking of getting myself a website to display my pictures and this was the cheapest thing I could find. I don't know about others, but Fir and Diliff, you guys have websites, do you think it is legit? Could you provide any better alternatives? Help from any other guild member is also appreciated. Thanks (talk)

=== NEW_DOCUMENT_MARKER ===

Thanks reading there now ! === NE